In [ ]:
import requests
from dotenv import load_dotenv
from pathlib import Path
import os
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pinecone import Pinecone,ServerlessSpec
from langchain_huggingface import HuggingFaceEmbeddings,ChatHuggingFace,HuggingFaceEndpoint
from langchain_pinecone import PineconeVectorStore
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI


In [ ]:
def get_hf_model():
    # use "mistralai/Mistral-7B-Instruct-v0.3" 
    # or "meta-llama/Llama-3.1-8B-Instruct"
    repo_id = "mistralai/Mistral-7B-Instruct-v0.3"

    llm = HuggingFaceEndpoint(
        repo_id=repo_id,
        task="text-generation", # <--- CRITICAL FIX: Add this line
        temperature=0.1,
        max_new_tokens=512,
        huggingfacehub_api_token=os.getenv("HUGGINGFACE_API_KEY")
    )
    return llm

In [2]:
%pip install PyPDF2

Note: you may need to restart the kernel to use updated packages.


In [3]:
env_path = Path.cwd() / ".env"
if not env_path.exists():
    env_path = Path.cwd() / "agents" / ".env"
load_dotenv(env_path)
OCR_SPACE_API_KEY = os.getenv("OCR_SPACE_API_KEY")

In [4]:
#For printed PDF uploads

def load_pdf_text(pdf_url):
    loader = PyMuPDFLoader(pdf_url)
    docs = loader.load()

    full_text = "\n".join([doc.page_content for doc in docs])
    return full_text


pdf_url = "https://your-imagekit-url/teacher_notes.pdf"

text = load_pdf_text(pdf_url)
print(text)

ConnectionError: HTTPSConnectionPool(host='your-imagekit-url', port=443): Max retries exceeded with url: /teacher_notes.pdf (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x00000235F85D6120>: Failed to resolve 'your-imagekit-url' ([Errno 11001] getaddrinfo failed)"))

In [5]:

##OCR TOOL --> if
def OCR_image_to_text(image_url):
    payload = {
        "apikey": OCR_SPACE_API_KEY,
        "url": image_url,
        "language": "eng",
        "isOverlayRequired": False,
        "OCREngine": 2
    }

    response = requests.post(
        "https://api.ocr.space/parse/image",
        data=payload
    )

    if response.status_code != 200:
        raise Exception(f"OCR API request failed: {response.status_code}")

    result = response.json()

    if result.get("IsErroredOnProcessing"):
        raise Exception(result.get("ErrorMessage"))

    parsed_results = result.get("ParsedResults", [])

    if not parsed_results:
        return "No text detected."

    extracted_text = parsed_results[0].get("ParsedText", "")

    if not extracted_text.strip():
        return "No readable text found."

    return extracted_text


In [ ]:
# Example
image_url = "https://ik.imagekit.io/k3p6avtbf/answer_scripts/Screenshot_2026-05-16_190504_zTe5mfpeJT.png"



In [ ]:
#text cleaning
import re

def clean_ocr_text(text):
    if not text:
        return ""

    # Remove non-printable characters
    text = re.sub(r'[\x00-\x1F\x7F-\x9F]', ' ', text)

    # Replace common OCR mistakes
    replacements = {
        '|': 'I',
        '¦': 'I',
        'ﬁ': 'fi',
        'ﬂ': 'fl',
        '¢': 'c',
        '©': 'o',
        '®': '',
        '™': '',
    }

    for wrong, correct in replacements.items():
        text = text.replace(wrong, correct)

    # Remove excessive punctuation junk
    text = re.sub(r'[~`^*_<>]+', ' ', text)

    # Fix hyphenated line breaks
    text = re.sub(r'-\s*\n\s*', '', text)

    # Merge broken lines inside sentences
    text = re.sub(r'(?<![.!?])\n(?!\n)', ' ', text)

    # Preserve paragraph breaks
    text = re.sub(r'\n{2,}', '\n\n', text)

    # Remove repeated spaces
    text = re.sub(r'[ \t]+', ' ', text)

    # Remove repeated punctuation
    text = re.sub(r'([.,!?])\1+', r'\1', text)

    # Remove isolated junk characters
    text = re.sub(r'\b[a-zA-Z]{1}\b(?=\s+[^\n])', '', text)

    # Clean spacing around punctuation
    text = re.sub(r'\s+([.,!?;:])', r'\1', text)

    return text.strip()

In [ ]:
#text spitting
def split_text(text):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=20,
        chunk_overlap=3,
        separators=["\n\n", "\n", ".", " ", ""]
    )

    chunks = splitter.split_text(text)
    return chunks

In [17]:
def get_embedding_model():

    embedding = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

    return embedding


In [18]:
embedding=get_embedding_model()


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
INDEX_NAME = os.getenv("PINECONE_INDEX_NAME", "aiss-aes-index")


# PINECONE INITIALIZATION
pc = Pinecone(api_key=PINECONE_API_KEY)

if INDEX_NAME not in pc.list_indexes().names():
    pc.create_index(
        name=INDEX_NAME,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

print(f"Using Pinecone index: {INDEX_NAME}")




Using Pinecone index: aiss-aes-index


In [ ]:
index = pc.Index(INDEX_NAME)
vector_store = PineconeVectorStore(
    index=index,
    embedding=embedding,
)

NameError: name 'pc' is not defined

In [11]:
def store_teacher_chunks(chunks, question_no, content_type, subject, exam_id, NAMESPACE):
    docs = []
    
    for chunk in chunks:
        docs.append(
            Document(
                page_content=chunk,
                metadata={
                    "type": content_type, # Use the variable instead of hardcoding "notes"
                    "subject": subject,
                    "question_no": question_no,
                    "exam_id": exam_id
                }
            )
        )

    # CRITICAL: Pass the namespace here so it actually goes to "midterm_2024"
    vector_store.add_documents(docs, namespace=NAMESPACE)
    print(f"Stored successfully in namespace: {NAMESPACE}")

In [20]:
teacher_chunks = [
    "Photosynthesis is the process by which green plants prepare food.",
    "Chlorophyll captures sunlight energy.",
    "Carbon dioxide and water are converted into glucose and oxygen."
]
store_teacher_chunks(teacher_chunks, question_no=1, content_type="answer_key", subject="maths", exam_id="midterm_2024", NAMESPACE="midterm_2024")

Stored successfully in namespace: midterm_2024


In [21]:
# 2. FIXED RETRIEVER FUNCTION
def retrieve_relevant_notes_langchain(question_text, namespace, top_k=5):
    # Ensure the filter matches what you stored (content_type="answer_key")
    retriever = vector_store.as_retriever(
        search_type='mmr',
        search_kwargs={
            "k": top_k,
            "namespace": namespace,
            "filter": {"type": "answer_key"} 
        }
    )

    docs = retriever.invoke(question_text)
    return [doc.page_content for doc in docs]

TILL HERE TEACHER UPLOADS NOW STUDENT AND QUESTIONS

In [22]:
# 3. EXECUTION FLOW
teacher_chunks = [
    "Photosynthesis is the process by which green plants prepare food.",
    "Chlorophyll captures sunlight energy."
]

# Store with content_type="answer_key" to match the retriever's filter
store_teacher_chunks(
    teacher_chunks, 
    question_no=1, 
    content_type="answer_key", 
    subject="biology", 
    exam_id="midterm_2024", 
    NAMESPACE="midterm_2024"
)


Stored successfully in namespace: midterm_2024


In [23]:

# Test the retrieval
question = "What is photosynthesis?"
results = retrieve_relevant_notes_langchain(question, namespace="midterm_2024")
print(results)

['Photosynthesis is the process by which green plants prepare food.', 'Carbon dioxide and water are converted into glucose and oxygen.', 'Chlorophyll captures sunlight energy.', 'Photosynthesis is the process by which green plants prepare food.', 'Photosynthesis is the process by which green plants prepare food.']


In [24]:
from langchain_core.prompts import ChatPromptTemplate

eval_prompt = ChatPromptTemplate.from_template("""
You are an expert academic evaluator. Your task is to grade a student's answer based on the provided Teacher's Answer Key and additional Contextual Notes.

### TEACHER'S ANSWER KEY:
{answer_key}

### CONTEXTUAL NOTES:
{context_notes}

### STUDENT'S ANSWER:
{student_answer}

### EVALUATION CRITERIA:
1. Accuracy: Does the answer align with the Teacher's Key?
2. Completeness: Does the student use relevant details found in the Contextual Notes?
3. Clarity: Is the explanation easy to understand?

### OUTPUT FORMAT:
- Score: [0 to 10]
- Strengths: [What they got right]
- Weaknesses: [What was missing or incorrect]
- Corrective Feedback: [How to improve]
""")

In [25]:
def get_gemini():   
    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash", # Or "gemini-2.5-pro" for complex grading
        temperature=0.1,         # Low temperature for objective evaluation
        google_api_key=os.getenv("GOOGLE_API_KEY")
    )
    return llm

In [26]:
def grade_student_submission(question, student_text, namespace):
    # 1. Retrieve notes (your existing function)
    notes = retrieve_relevant_notes_langchain(question, namespace=namespace, top_k=3)
    context_str = "\n".join(notes)
    
    # 2. Retrieve teacher answer key (using a specific filter)
    # Assuming you store the "key" with a specific metadata type
    key_docs = vector_store.as_retriever(
        search_kwargs={"k": 1, "namespace": namespace, "filter": {"type": "answer_key"}}
    ).invoke(question)
    answer_key_str = key_docs[0].page_content if key_docs else "No key provided."

    # 3. Run the LLM
    llm=get_gemini() 
    chain = eval_prompt | llm
    
    response = chain.invoke({
        "answer_key": answer_key_str,
        "context_notes": context_str,
        "student_answer": student_text
    })
    
    return response.content

# Example usage:
student_ans = "Photosynthesis is how plants make food using sun and water."
grading_result = grade_student_submission("What is photosynthesis?", student_ans, "midterm_2024")
print(grading_result)

- Score: 7/10
- Strengths:
    *   Accurately identifies the core purpose of photosynthesis: plants making food.
    *   Correctly identifies two crucial components/inputs: sun (sunlight) and water.
    *   The explanation is clear, concise, and easy to understand.
- Weaknesses:
    *   Missed the specificity of "green" plants.
    *   Did not mention carbon dioxide as another essential input.
    *   Did not specify the products of photosynthesis (glucose and oxygen).
    *   Did not mention chlorophyll, the pigment responsible for capturing sunlight.
- Corrective Feedback:
    To improve your answer, try to include more specific details about the process. Remember that photosynthesis specifically occurs in **green** plants. Think about *all* the ingredients plants need (inputs) and what they produce (outputs). For a more complete answer, you could say: "Photosynthesis is how **green** plants make food (**glucose**) using **carbon dioxide**, water, and **sunlight** (captured by **chlo